## StayNest — Delta Lake & Lakehouse Engineering

An end-to-end Data Engineering project demonstrating Lakehouse architecture using Databricks, PySpark and Delta Lake.

This project processes StayNest hotel booking data through a Medallion Architecture and demonstrates core Lakehouse engineering patterns, including:

- Broadcast joins and Spark query optimization
- Delta Lake table creation and transactional operations
- UPDATE and DELETE operations
- Delta Lake Time Travel and RESTORE
- OPTIMIZE and ZORDER
- Bronze, Silver and Gold data layers
- Data enrichment and transformation with PySpark
- Incremental data ingestion using MERGE
- Validation of row counts and transformed datasets
### Architecture
Raw Booking Data
       │
       ▼
   Bronze Layer
  Raw + Metadata
       │
       ▼
   Silver Layer
 Completed + Enriched
       │
       ▼
    Gold Layer
 City Revenue Metrics
### Technology Stack

Databricks · PySpark · Delta Lake · Spark SQL · Lakehouse Architecture

### Project Objective

The objective is to demonstrate how raw booking data can be transformed into reliable analytical datasets while applying practical Lakehouse engineering techniques for data quality, optimization, historical versioning and incremental processing.

## 1. Project Setup & Data Validation

The project uses three source datasets stored in a Databricks Volume:

bookings.csv — hotel booking transactions
hotels.csv — hotel dimension data
bookings_updates.csv — incremental booking updates

The notebook configures the Databricks catalog, schema and source volume, then validates the input datasets before processing begins.

Input Data Validation

Expected source volumes:

Dataset	Expected Rows
Bookings	12,000
Hotels	200
Booking Updates	200

These checks provide an initial validation of the source data before the Lakehouse processing pipeline begins.

In [0]:
BASE    = "/Volumes/workspace/default/staynest"
CATALOG = "workspace"
SCHEMA  = "default"
FQN = lambda name: f"{CATALOG}.{SCHEMA}.{name}"

read_csv = lambda name: (spark.read
    .option("header", True).option("inferSchema", True)
    .csv(f"{BASE}/{name}.csv"))

bookings_df = read_csv("bookings")
hotels_df   = read_csv("hotels")
updates_df  = read_csv("bookings_updates")

print(f"bookings: {bookings_df.count()}, hotels: {hotels_df.count()}, "
      f"updates: {updates_df.count()}")

## 2. Broadcast Join & Query Plan Optimization

The bookings dataset is significantly larger than the hotel dimension. Since the hotel dataset is small, it can be broadcast to the worker nodes so Spark can perform the join without repartitioning the large bookings dataset.

Two physical plans are inspected:

A standard join, allowing Spark to choose the join strategy.
An explicit broadcast join using broadcast(hotels_df).

The hotel city column is removed before the join because the bookings dataset already contains a city column.

Engineering Takeaway

A broadcast join is useful when one side of a join is sufficiently small to fit in executor memory. Broadcasting the smaller dataset allows Spark to perform a broadcast hash join and avoids shuffling the larger dataset across the cluster.

In this project, Spark's optimizer may automatically choose the broadcast strategy because the hotel dimension contains only 200 rows. Therefore, the normal and explicitly broadcasted plans can be the same.

In [0]:
from pyspark.sql.functions import broadcast

# Normal join - let Spark choose the join strategy
normal_join = bookings_df.join(
    hotels_df.drop("city"),
    "hotel_id"
)

print("=== Normal Join Physical Plan ===")
normal_join.explain()

# Forced broadcast join
broadcast_join = bookings_df.join(
    broadcast(hotels_df.drop("city")),
    "hotel_id"
)

print("=== Broadcast Join Physical Plan ===")
broadcast_join.explain()

# Explanation:
# Spark/Photon automatically selected a BroadcastHashJoin because
# hotels_df is small. The explicit broadcast() confirms the same
# strategy. Broadcasting the small dimension avoids shuffling the
# larger bookings dataset across the join key.


## 3. Delta Lake Table & Transaction History

The booking dataset is persisted as a **managed Delta Lake table**. Delta Lake provides ACID transactions and maintains a transaction log that records changes made to the table over time.

The table is then modified using two transactional operations:

* **UPDATE** — changes bookings with `pending` status to `completed`.
* **DELETE** — removes bookings with `cancelled` status.

After these operations, `DESCRIBE HISTORY` is used to inspect the Delta transaction history and verify that each operation was recorded as a separate versioned commit.

### Engineering Takeaway

Delta Lake maintains an immutable transaction log that enables reliable data modifications and historical tracking. Each write or transactional operation creates a new table version, providing the foundation for capabilities such as **Time Travel, auditing and recovery**.

In [0]:
bookings_table = FQN("bookings_delta")

bookings_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(bookings_table)

print(f"Created Delta table: {bookings_table}")
print(f"Initial row count: {spark.table(bookings_table).count()}")

# Update pending bookings to completed
spark.sql(f"""
    UPDATE {bookings_table}
    SET status = 'completed'
    WHERE status = 'pending'
""")

# Check status distribution
spark.sql(f"""
    SELECT status, COUNT(*) AS row_count
    FROM {bookings_table}
    GROUP BY status
    ORDER BY status
""").show()

# Delete cancelled bookings
spark.sql(f"""
    DELETE FROM {bookings_table}
    WHERE status = 'cancelled'
""")

print(f"Row count after UPDATE + DELETE: {spark.table(bookings_table).count()}")

# Show Delta transaction history
spark.sql(f"""
    DESCRIBE HISTORY {bookings_table}
""").show(truncate=False)

## 4. Delta Lake Time Travel & RESTORE

Delta Lake maintains historical versions of a table through its transaction log. This allows previous versions of the dataset to be queried without modifying the current table.

In this step:

1. The table is read using **Time Travel** at version `0`, representing the original 12,000-row dataset before the subsequent `UPDATE` and `DELETE` operations.
2. The table is restored to version `0` using `RESTORE TABLE`.
3. The current row count is validated after the restore.
4. The transaction history is inspected to confirm that the `RESTORE` operation created a new Delta commit.

### Engineering Takeaway

**Time Travel** provides read-only access to historical table versions, while **RESTORE** changes the current table state to match a previous version.

This provides a practical recovery mechanism when a table needs to be rolled back after an incorrect or unwanted data modification.


In [0]:


# Read the table as it existed at version 0
version0_df = (
    spark.read
    .option("versionAsOf", 0)
    .table(bookings_table)
)

print("Version 0 row count:", version0_df.count())

# Restore the current table back to version 0
spark.sql(f"""
    RESTORE TABLE {bookings_table}
    TO VERSION AS OF 0
""")

# Check the current table after RESTORE
print("Current row count after RESTORE:", spark.table(bookings_table).count())

# Show the latest Delta history
spark.sql(f"""
    DESCRIBE HISTORY {bookings_table}
""").show(truncate=False)


## 5. Delta Lake Optimization: OPTIMIZE & ZORDER

Delta Lake tables can accumulate many small data files as data is written and modified. The `OPTIMIZE` command compacts these files into larger, more efficient files, reducing file-management overhead during queries.

The table is also optimized using:

`ZORDER BY (city)`

Z-Ordering colocates related values within the underlying data files, improving **data skipping** when queries frequently filter on the selected column.

### Why `city`?

`city` is a useful Z-Ordering column because the dataset contains multiple cities and analytical queries may commonly filter or group bookings by city. Organizing data around city values can therefore improve selective queries.

### Why not `status`?

`status` has only a few distinct values such as `completed`, `pending` and `cancelled`. Because of this low cardinality, Z-Ordering on `status` provides less opportunity for effective data skipping compared with a column containing many more distinct values.

### Engineering Takeaway

`OPTIMIZE` focuses on **file compaction**, while `ZORDER` helps organize data to improve **data skipping for selective queries**. The choice of Z-Order columns should be driven by expected query/filter patterns and data distribution.


In [0]:


# Compact small Delta files
spark.sql(f"""
    OPTIMIZE {bookings_table}
""")

# Compact and organize data using city for data skipping
spark.sql(f"""
    OPTIMIZE {bookings_table}
    ZORDER BY (city)
""")

print("OPTIMIZE and ZORDER completed successfully.")

# Show the latest Delta history entries
spark.sql(f"""
    DESCRIBE HISTORY {bookings_table}
""").select(
    "version",
    "operation",
    "operationParameters",
    "operationMetrics"
).show(truncate=False)


## 6. Bronze Layer — Raw Data Ingestion

The Bronze layer is the landing zone for raw source data. Its purpose is to preserve the incoming dataset with minimal transformation while adding ingestion metadata for operational tracking.

The raw booking data is written to a **Delta table** named `bronze_bookings`.

The Bronze layer:

* Preserves all source booking records.
* Retains the original source columns and values.
* Adds an `ingested_at` timestamp to record when the data entered the Lakehouse.
* Uses Delta Lake as the storage format for reliable downstream processing.

### Engineering Takeaway

The Bronze layer provides a reliable historical landing point between source systems and downstream transformations. Keeping the raw data available makes it possible to trace and validate subsequent Silver and Gold transformations.


In [0]:


from pyspark.sql.functions import current_timestamp

bronze_table = FQN("bronze_bookings")

bronze_df = (
    bookings_df
    .withColumn("ingested_at", current_timestamp())
)

bronze_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(bronze_table)

print(f"Created Bronze table: {bronze_table}")
print(f"Bronze row count: {spark.table(bronze_table).count()}")

# Inspect the Bronze data
spark.table(bronze_table).show(5, truncate=False)

# Check the schema
spark.table(bronze_table).printSchema()


## 7. Silver Layer — Cleaning & Enrichment

The Silver layer transforms the raw Bronze data into a cleaner, analysis-ready dataset.

The `silver_bookings` table is created from the Bronze layer by:

* Filtering the dataset to **completed bookings**.
* Joining bookings with the hotel dimension using `hotel_id`.
* Enriching each booking with hotel attributes:

  * `hotel_name`
  * `category`
  * `star_rating`
* Removing the duplicate `city` column from the hotel dimension so the resulting dataset contains a single `city` field.

### Data Quality Observation

The Bronze layer contains **10,563 completed bookings**, while the resulting Silver layer contains **9,660 records**.

The difference is caused by the inner join: **903 completed bookings do not have a matching `hotel_id` in the hotel dimension**.

This demonstrates an important data-engineering consideration: referential integrity between fact-style transactional data and dimension data must be validated during enrichment.

### Engineering Takeaway

The Silver layer applies business rules, filtering and dimensional enrichment while producing a consistent dataset for downstream analytical processing.


In [0]:


from pyspark.sql.functions import col

silver_table = FQN("silver_bookings")

silver_df = (
    spark.table(bronze_table)
    .filter(col("status") == "completed")
    .join(
        hotels_df.drop("city"),
        on="hotel_id",
        how="inner"
    )
)

silver_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(silver_table)

print(f"Created Silver table: {silver_table}")
print(f"Silver row count: {spark.table(silver_table).count()}")

# Inspect the Silver data
spark.table(silver_table).show(5, truncate=False)

# Check schema
spark.table(silver_table).printSchema()


In [0]:


spark.sql(f"""
    SELECT
        status,
        COUNT(*) AS row_count
    FROM {silver_table}
    GROUP BY status
""").show()

print("Silver columns:")
print(spark.table(silver_table).columns)

## 8. Gold Layer — Business-Ready Revenue Metrics

The Gold layer contains curated datasets designed for reporting, analytics and business consumption.

Using the Silver layer, the project creates a `gold_city_revenue` Delta table containing:

* Number of completed bookings per city.
* Total booking revenue per city.
* Results ordered by total revenue in descending order.

### Business Purpose

This aggregation provides a simple city-level view of booking performance that could be consumed by a BI dashboard, reporting process or downstream analytical workload.

### Engineering Takeaway

The Gold layer reduces the complexity of the underlying transactional data into a business-ready dataset. Instead of repeatedly processing individual booking records, downstream consumers can query a small, purpose-built aggregate.

In this project, the Silver layer contains **9,660 enriched bookings**, which are aggregated into **10 city-level records** in the Gold layer.


In [0]:


from pyspark.sql.functions import count, sum, col

gold_table = FQN("gold_city_revenue")

gold_df = (
    spark.table(silver_table)
    .groupBy("city")
    .agg(
        count("booking_id").alias("bookings"),
        sum("amount").alias("revenue")
    )
    .orderBy(col("revenue").desc())
)

gold_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(gold_table)

print(f"Created Gold table: {gold_table}")
print(f"Gold row count: {spark.table(gold_table).count()}")

# Display business-ready result
spark.table(gold_table).show(truncate=False)


## 9. Incremental Load with Delta MERGE

The project simulates an incremental daily ingestion batch containing both existing records that have changed and completely new bookings.

The `MERGE` operation applies this batch to the `bookings_delta` table in a single atomic operation.

### Incremental Batch

The update dataset contains:

* **150 existing booking IDs** requiring updates.
* **50 new booking IDs** requiring inserts.
* **200 total records** in the incoming batch.

The `MERGE` logic matches records using `booking_id`:

* **Matched records** → update the existing booking.
* **Unmatched records** → insert the new booking.

### Validation

The target table initially contains **12,000 records**.

After the incremental merge, it contains **12,050 records**.

The increase of **50 records** confirms that the 50 new bookings were inserted while the 150 existing bookings were updated rather than duplicated.

### Engineering Takeaway

Delta Lake `MERGE` provides an efficient pattern for incremental **upsert** processing. It allows updates and inserts to be handled together while maintaining transactional consistency.

This pattern is commonly used in Lakehouse pipelines where new batches contain a mixture of new records and changes to existing records.


In [0]:


before_count = spark.table(bookings_table).count()

print(f"Row count before MERGE: {before_count}")

# Register today's batch as a temporary view
updates_df.createOrReplaceTempView("bookings_updates_batch")

# MERGE changed bookings and new bookings
spark.sql(f"""
    MERGE INTO {bookings_table} AS target
    USING bookings_updates_batch AS source
    ON target.booking_id = source.booking_id

    WHEN MATCHED THEN
        UPDATE SET *

    WHEN NOT MATCHED THEN
        INSERT *
""")

after_count = spark.table(bookings_table).count()

print(f"Row count after MERGE: {after_count}")
print(f"Rows added: {after_count - before_count}")


## 10. Final Validation & Project Summary

### Pipeline Results

| Layer / Component                     | Result |
| ------------------------------------- | -----: |
| Source bookings                       | 12,000 rows |
| Source hotels                         | 200 rows |
| Incremental updates                   | 200 rows |
| Bronze bookings                       | 12,000 rows |
| Silver bookings                       | 9,660 rows |
| Completed bookings without hotel match| 903 |
| Gold city records                     | 10 |
| Delta table before MERGE              | 12,000 rows |
| Delta table after MERGE               | 12,050 rows |

### Lakehouse Flow

```text
                 ┌─────────────────────┐
                 │   Raw CSV Sources   │
                 │ Bookings / Hotels   │
                 │ Incremental Updates │
                 └──────────┬──────────┘
                            │
                            ▼
                 ┌─────────────────────┐
                 │   Bronze Layer      │
                 │   12,000 bookings   │
                 │   + ingested_at     │
                 └──────────┬──────────┘
                            │
                            ▼
                 ┌─────────────────────┐
                 │    Silver Layer     │
                 │ Completed bookings  │
                 │ + hotel enrichment  │
                 │    9,660 records    │
                 └──────────┬──────────┘
                            │
                            ▼
                 ┌─────────────────────┐
                 │     Gold Layer      │
                 │ City-level revenue  │
                 │    10 cities        │
                 └─────────────────────┘

        Delta Lake Operations
        ─────────────────────
        UPDATE / DELETE
        Time Travel / RESTORE
        OPTIMIZE / ZORDER
        Incremental MERGE
```

### Key Engineering Concepts Demonstrated

This project demonstrates practical Lakehouse engineering patterns using **Databricks, PySpark, Spark SQL and Delta Lake**:

* Medallion Architecture
* Delta Lake transactional processing
* Broadcast join optimization
* Delta transaction history
* Time Travel and table recovery
* File compaction with `OPTIMIZE`
* Data organization with `ZORDER`
* Bronze-to-Silver transformation
* Dimensional enrichment
* Gold-layer business aggregation
* Incremental upsert processing with `MERGE`
* Basic data-quality and referential-integrity validation

### Conclusion

The StayNest pipeline demonstrates how raw transactional data can be progressively transformed into reliable, enriched and business-ready datasets while using Delta Lake capabilities for **transaction management, historical versioning, optimization and incremental processing**.
